In [1]:
import pandas as pd
import swifter
import aiohttp
import asyncio
import requests
import json
import gzip


/home/ary2260/miniconda3/envs/prithvi-env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:23: UserWarning: You are using pyarrow version 13.0.0 which is known to be insecure. See https://www.cve.org/CVERecord?id=CVE-2023-47248 for further details. Please upgrade to pyarrow>=14.0.1 or install pyarrow-hotfix to patch your current version.
  warnings.warn(


In [2]:
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import aiohttp
import asyncio
import requests

import nest_asyncio
nest_asyncio.apply()


def get_json_async_fn(url_fn, processing_fn):
    async def json_async_fn(*input_args):
        url = url_fn(*input_args)
        async with aiohttp.ClientSession() as session:
            async with session.get(url) as response:
                data = await response.json()
                return processing_fn(data, *input_args)
    return json_async_fn

def get_status_async_fn(url_fn, processing_fn):
    async def status_async_fn(*input_args):
        url = url_fn(*input_args)
        async with aiohttp.ClientSession() as session:
            async with session.get(url) as response:
                tmp = await response.read()
                data = response.status
                return processing_fn(data, *input_args)
    return status_async_fn

def run_async_on_event_loop_on_multiple_threads(fn, values, max_workers):
    def run_async_on_event_loop(fn, value):
        loop = asyncio.get_event_loop()
        results = loop.run_until_complete(fn(value))
        return results
    pairs = [(fn, x) for x in values]
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(tqdm(executor.map(run_async_on_event_loop, *zip(*pairs)), total=len(values), desc="Processing"))
    return list(results)

def run_on_multiple_threads(fn, values, max_workers):
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(tqdm(executor.map(fn, values), total=len(values), desc="Processing"))
    return list(results)

def get_optimized_json_processing_fn(url_fn, processing_fn, max_workers):
    json_async_fn = get_json_async_fn(url_fn, processing_fn)
    return lambda values: run_async_on_event_loop_on_multiple_threads(json_async_fn, values, max_workers)

def get_optimized_status_processing_fn(url_fn, processing_fn, max_workers):
    status_async_fn = get_status_async_fn(url_fn, processing_fn)
    return lambda values: run_async_on_event_loop_on_multiple_threads(status_async_fn, values, max_workers)

In [3]:
# check availibity of exp structures

def if_exp_url(entry_id):
    return f"https://www.ebi.ac.uk/pdbe/graph-api/uniprot/protvista/unipdb/{entry_id}"

def if_exp_processing(json_data, entry_id):
    if json_data:
        return True
    else:
        return False

if_exp_optimized_20 = get_optimized_json_processing_fn(if_exp_url, if_exp_processing, max_workers=20)

In [4]:
# check non-nmr avail

def non_nmr_url(entry_id):
    return f"https://www.ebi.ac.uk/pdbe/graph-api/uniprot/protvista/unipdb/{entry_id}"

def non_nmr_processing(json_data, entry_id):
    pdb_list = []
    for pdb_struct in json_data[entry_id]['tracks'][0]['data']:
        output = {'type': 'PDB'}
        pdb_id = pdb_struct['label']['id']
        output['id'] = pdb_id
        output['resolution'] = pdb_struct['label']['resolution']
        if not output['resolution']:
            output['resolution'] = None
        pdb_list.append(output)

    pdbs_df = pd.DataFrame(pdb_list)
    pdbs_df['id'] = pdbs_df['id'].str.upper()
    pdbs_df = pdbs_df.dropna()
    pdbs_df = pdbs_df.set_index('id', drop=False)
    if len(pdbs_df) == 0:
        return False
    return True

non_nmr_optimized_20 = get_optimized_json_processing_fn(non_nmr_url, non_nmr_processing, max_workers=20)
non_nmr_optimized_30 = get_optimized_json_processing_fn(non_nmr_url, non_nmr_processing, max_workers=30)

In [5]:
# get pdb list

def get_pdb_url(protein_id):
    return f"https://www.ebi.ac.uk/pdbe/graph-api/uniprot/protvista/unipdb/{protein_id}"

def get_pdbs_processing(json_data, protein_id):
    pdbs_list = []
    for pdb_struct in json_data[protein_id]['tracks'][0]['data']:
        pdb_id = pdb_struct['label']['id']
        pdbs_list.append(pdb_id)
    return pdbs_list

get_pdbs_optimized_20 = get_optimized_json_processing_fn(get_pdb_url, get_pdbs_processing, max_workers=20)

In [6]:
# check pdb availibity

def get_check_url(pdb_id):
    return f"https://www.ebi.ac.uk/pdbe/entry-files/download/pdb{pdb_id.lower()}.ent"

def get_check_avail_processing(status_code, pdb_id):
    if status_code == 200:
        return True
    elif status_code == 404:
        return False
    else:
        raise Exception(f"invalid status code {status_code}")

check_pdbs_avail_optimized_10 = get_optimized_status_processing_fn(get_check_url, get_check_avail_processing, max_workers=10)

In [7]:
# check entry validity

def check_entry_validity_part1(pdb_list):
    responses = check_pdbs_avail_optimized_10(pdb_list[:2])
    score = sum(responses)
    if score:
        return True
    else:
        return False

def check_entry_validity_part2(pdb_list):
    responses = check_pdbs_avail_optimized_10(pdb_list)
    score = sum(responses)
    if score:
        return True
    else:
        return False

check_entry_validity_part1_optimized_10 = lambda list_of_pdb_list: run_on_multiple_threads(check_entry_validity_part1, list_of_pdb_list, max_workers=10)
check_entry_validity_part2_optimized_10 = lambda list_of_pdb_list: run_on_multiple_threads(check_entry_validity_part2, list_of_pdb_list, max_workers=10)
check_entry_validity_part2_optimized_5 = lambda list_of_pdb_list: run_on_multiple_threads(check_entry_validity_part2, list_of_pdb_list, max_workers=5)

dataset

In [8]:
df_main = pd.read_csv("/home/ary2260/Work/DeepForest/customer_small/vevo/human_prote/full_filtered_human_prote/18337_genes_reviewed_human_prot_filtered.csv")

In [9]:
df_main = df_main[df_main['Gene Names (primary; single)']=='ITGAV']

In [10]:
df_main

,Entry,Entry Name,Protein names,Gene Names,Reviewed,Gene Names (primary),is_exp_present,has_non_nmr,entry_validity_part1,Gene Names (primary; single)
7640,P06756,ITAV_HUMAN,Integrin alpha-V (Vitronectin receptor) (Vitro...,ITGAV MSK8 VNRA VTNR,reviewed,ITGAV,True,True,False,ITGAV


### exp structures

In [11]:
df_main["is_exp_present"] = pd.Series(if_exp_optimized_20(df_main['Entry'].to_list()))

Processing: 100%|██████████| 1/1 [00:03<00:00,  3.69s/it]
/tmp/ipykernel_141663/111294344.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_main["is_exp_present"] = pd.Series(if_exp_optimized_20(df_main['Entry'].to_list()))


In [12]:
df_main['is_exp_present'].value_counts()

Series([], Name: count, dtype: int64)

In [13]:
df_exp_sub = df_main[df_main['is_exp_present']==True]

In [14]:
df_exp_sub = df_exp_sub.reset_index(drop=False)
df_exp_sub.columns[0]

'index'

### non nmr

In [15]:
df_exp_sub["has_non_nmr"] = pd.Series(non_nmr_optimized_30(df_exp_sub['Entry'].to_list()))

Processing: 0it [00:00, ?it/s]


In [16]:
df_exp_sub["has_non_nmr"].value_counts()

Series([], Name: count, dtype: int64)

In [17]:
# merge with main

df_main['has_non_nmr'] = pd.Series([False]*len(df_main))

/tmp/ipykernel_141663/1373894953.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_main['has_non_nmr'] = pd.Series([False]*len(df_main))


In [18]:
df_exp_sub = df_exp_sub.set_index('index')

In [19]:
df_exp_sub

,Entry,Entry Name,Protein names,Gene Names,Reviewed,Gene Names (primary),is_exp_present,has_non_nmr,entry_validity_part1,Gene Names (primary; single)
index,,,,,,,,,,


In [20]:
df_main.update(df_exp_sub)

In [21]:
df_main

,Entry,Entry Name,Protein names,Gene Names,Reviewed,Gene Names (primary),is_exp_present,has_non_nmr,entry_validity_part1,Gene Names (primary; single)
7640,P06756,ITAV_HUMAN,Integrin alpha-V (Vitronectin receptor) (Vitro...,ITGAV MSK8 VNRA VTNR,reviewed,ITGAV,NaN,NaN,False,ITGAV


In [22]:
df_main['has_non_nmr'].value_counts()

Series([], Name: count, dtype: int64)

In [24]:
# df_main.to_csv("non_nmr_checkpoint.csv", index=False)

In [23]:
df_main

,Entry,Entry Name,Protein names,Gene Names,Reviewed,Gene Names (primary),is_exp_present,has_non_nmr,entry_validity_part1,Gene Names (primary; single)
7640,P06756,ITAV_HUMAN,Integrin alpha-V (Vitronectin receptor) (Vitro...,ITGAV MSK8 VNRA VTNR,reviewed,ITGAV,NaN,NaN,False,ITGAV


pdblist check

In [24]:
df_next_sub = df_main[df_main['has_non_nmr']==True]
df_next_sub = df_next_sub.reset_index(drop=False)
df_next_sub.columns[0]

'index'

In [25]:
df_next_sub["pdb_list"] = pd.Series(get_pdbs_optimized_20(df_next_sub['Entry'].to_list()))


Processing: 0it [00:00, ?it/s]


In [36]:
K = get_pdbs_optimized_20(['P06756'])

Processing: 100%|██████████| 1/1 [00:02<00:00,  2.79s/it]


In [39]:
for id in K[0]:
    print(f"https://www.ebi.ac.uk/pdbe/entry-files/download/pdb{id.lower()}.ent")

https://www.ebi.ac.uk/pdbe/entry-files/download/pdb9czd.ent
https://www.ebi.ac.uk/pdbe/entry-files/download/pdb9cza.ent
https://www.ebi.ac.uk/pdbe/entry-files/download/pdb5ffg.ent
https://www.ebi.ac.uk/pdbe/entry-files/download/pdb8tcg.ent
https://www.ebi.ac.uk/pdbe/entry-files/download/pdb9ind.ent
https://www.ebi.ac.uk/pdbe/entry-files/download/pdb6ujb.ent
https://www.ebi.ac.uk/pdbe/entry-files/download/pdb9czf.ent
https://www.ebi.ac.uk/pdbe/entry-files/download/pdb6ujc.ent
https://www.ebi.ac.uk/pdbe/entry-files/download/pdb9cz7.ent
https://www.ebi.ac.uk/pdbe/entry-files/download/pdb8tcf.ent
https://www.ebi.ac.uk/pdbe/entry-files/download/pdb8vs6.ent
https://www.ebi.ac.uk/pdbe/entry-files/download/pdb4um8.ent
https://www.ebi.ac.uk/pdbe/entry-files/download/pdb4um9.ent
https://www.ebi.ac.uk/pdbe/entry-files/download/pdb4g1m.ent
https://www.ebi.ac.uk/pdbe/entry-files/download/pdb6uja.ent
https://www.ebi.ac.uk/pdbe/entry-files/download/pdb7y1t.ent
https://www.ebi.ac.uk/pdbe/entry-files/d

In [42]:
pd.Series(check_entry_validity_part2_optimized_10(K))


Processing: 100%|██████████| 1/1 [00:11<00:00, 11.69s/it]


0    True
dtype: bool

In [26]:
df_next_sub["entry_validity_part1"] = pd.Series(check_entry_validity_part1_optimized_10(df_next_sub["pdb_list"].to_list()))


Processing: 0it [00:00, ?it/s]


In [27]:
df_next_sub.value_counts("entry_validity_part1")

Series([], Name: count, dtype: int64)

In [35]:
# df_next_sub.to_csv("checkpoint_next_sub_nmr.csv")

In [11]:
# df_next_sub = pd.read_csv("/home/ubuntu/vevo/vevo_all/checkpoint_next_sub_nmr.csv")

In [12]:
df_next_next_sub = df_next_sub[df_next_sub['entry_validity_part1']==False]
df_next_next_sub = df_next_next_sub.reset_index(drop=False)
df_next_next_sub.columns[0]

'level_0'

In [ ]:
df_next_next_sub["entry_validity_part1"] = pd.Series(check_entry_validity_part2_optimized_5(df_next_next_sub["pdb_list"].to_list()))

In [17]:
df_next_next_sub["entry_validity_part1"].value_counts()

entry_validity_part1
False    468
Name: count, dtype: int64

In [28]:
df_next_sub = df_next_sub.set_index('index')

In [29]:
df_next_sub

,Entry,Entry Name,Protein names,Gene Names,Reviewed,Gene Names (primary),is_exp_present,has_non_nmr,entry_validity_part1,Gene Names (primary; single),pdb_list
index,,,,,,,,,,,


In [30]:
df_main['entry_validity_part1'] = pd.Series([False]*len(df_main))

/tmp/ipykernel_141663/3789242215.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_main['entry_validity_part1'] = pd.Series([False]*len(df_main))


In [31]:
df_main.update(df_next_sub)

In [32]:
df_main.value_counts("entry_validity_part1")

Series([], Name: count, dtype: int64)

In [33]:
len(df_main)

1

In [34]:
df_main

,Entry,Entry Name,Protein names,Gene Names,Reviewed,Gene Names (primary),is_exp_present,has_non_nmr,entry_validity_part1,Gene Names (primary; single)
7640,P06756,ITAV_HUMAN,Integrin alpha-V (Vitronectin receptor) (Vitro...,ITGAV MSK8 VNRA VTNR,reviewed,ITGAV,NaN,NaN,NaN,ITGAV


In [25]:
df_main.columns

Index(['Entry', 'Entry Name', 'Protein names', 'Gene Names', 'Reviewed',
       'Gene Names (primary)', 'is_exp_present', 'has_non_nmr',
       'entry_validity_part1'],
      dtype='object')

In [26]:
df_main['Gene Names (primary; single)'] = df_main['Gene Names (primary)'].apply(lambda x: x.split('; ')[0].strip())

In [27]:
df_main.to_csv("18337_genes_reviewed_human_prot_filtered.csv", index=False)